In [5]:
# STEP 1: Import Necessary Libraries
# ==============================================================================
import os
import sys
import subprocess
import joblib
import warnings
import numpy as np
import pandas as pd

# Data preprocessing and evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Regression models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')
print("Step 1: Libraries imported successfully.")

Step 1: Libraries imported successfully.


In [6]:
# STEP 2: Load and Clean Dataset
file_path = r"C:\Users\BM\Documents\lab-work\house-price-predictor\houses_improved.csv"
target_column = 'Price_ETB'

# Load dataset using standard UTF-8 encoding
df = pd.read_csv(file_path, encoding='utf-8')

# Remove rows where the target value is missing
df = df.dropna(subset=[target_column])

# Separate feature matrix (X) and target vector (y)
X = df.drop(columns=[target_column])
y = df[target_column]

# Categorize numerical and categorical feature names
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Step 2: Dataset loaded. Total Shape: {df.shape}")
print(f"Features: {len(num_cols)} Numeric | {len(cat_cols)} Categorical")

Step 2: Dataset loaded. Total Shape: (1000, 12)
Features: 7 Numeric | 4 Categorical


In [7]:
# STEP 3: Split Data into Train and Test Sets
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Step 3: Data split into Training ({X_train.shape[0]}) and Testing ({X_test.shape[0]}) sets.")

Step 3: Data split into Training (800) and Testing (200) sets.


In [8]:
# ==============================================================================
# STEP 4: Define Machine Learning Pipelines & Combinations (Grid Search)
# ==============================================================================
from IPython.display import display

encoders = {
    'OneHot': OneHotEncoder(handle_unknown='ignore', sparse_output=False),
    'Ordinal': OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
}

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(),
    'SVR': SVR(C=1e7, epsilon=1.0),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'KNN': KNeighborsRegressor()
}

results = []
combo_id = 1

# Evaluate all 42 combinations (2 Encoders x 3 Scalers x 7 Regressors)
for enc_name, encoder in encoders.items():
    for scale_name, scaler in scalers.items():
        for model_name, model in models.items():
            
            transformers = []
            if num_cols:
                transformers.append(('num', scaler, num_cols))
            if cat_cols:
                transformers.append(('cat', encoder, cat_cols))
            
            preprocessor = ColumnTransformer(transformers=transformers)
            pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('regressor', model)
            ])
            
            try:
                pipeline.fit(X_train, y_train)
                y_pred = pipeline.predict(X_test)
                
                r2 = r2_score(y_test, y_pred)
                rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                mae = mean_absolute_error(y_test, y_pred)
                
                results.append({
                    'ID': combo_id,
                    'Encoder': enc_name,
                    'Scaler': scale_name,
                    'Algorithm': model_name,
                    'R2_Score': round(r2, 4),
                    'RMSE': round(rmse, 4),
                    'MAE': round(mae, 4)
                })
            except Exception as e:
                print(f"Error in combination {combo_id}: {e}")
            
            combo_id += 1

# Rank results by performance (R2 Score)
results_df = pd.DataFrame(results).sort_values(by='R2_Score', ascending=False).reset_index(drop=True)

# Display the summary table directly below the notebook cell
print("Step 4: Evaluation finished across all 42 combinations.\n")
print("📊 Complete Model Evaluation Summary (Ranked by R2 Score):")
display(results_df)

Step 4: Evaluation finished across all 42 combinations.

📊 Complete Model Evaluation Summary (Ranked by R2 Score):


,ID,Encoder,Scaler,Algorithm,R2_Score,RMSE,MAE
0,31,Ordinal,MinMaxScaler,SVR,0.9318,336092.5572,256425.7595
1,38,Ordinal,RobustScaler,SVR,0.9257,350788.6033,270821.5919
2,10,OneHot,MinMaxScaler,SVR,0.9237,355408.2770,275508.5927
3,3,OneHot,StandardScaler,SVR,0.9225,358352.1156,280191.4815
4,17,OneHot,RobustScaler,SVR,0.9107,384715.6381,296866.5923
5,24,Ordinal,StandardScaler,SVR,0.9047,397222.6066,310704.6996
6,13,OneHot,MinMaxScaler,GradientBoosting,0.9036,399634.5812,301645.6488
7,6,OneHot,StandardScaler,GradientBoosting,0.9036,399688.0859,301699.1168
8,20,OneHot,RobustScaler,GradientBoosting,0.9033,400253.1779,301847.3975
9,27,Ordinal,StandardScaler,GradientBoosting,0.8972,412657.8603,318469.8938


In [9]:
# STEP 5: Re-train Winning Pipeline & Save Model Bundle
# ==============================================================================
# Extract best parameters from results
best_enc_name = results_df.iloc[0]['Encoder']
best_scale_name = results_df.iloc[0]['Scaler']
best_model_name = results_df.iloc[0]['Algorithm']

best_preprocessor = ColumnTransformer(transformers=[
    ('num', scalers[best_scale_name], num_cols),
    ('cat', encoders[best_enc_name], cat_cols)
])

winning_pipeline = Pipeline(steps=[
    ('preprocessor', best_preprocessor),
    ('regressor', models[best_model_name])
])

winning_pipeline.fit(X_train, y_train)

# Save the fitted pipeline and feature list together as a bundle
model_bundle = {
    'model': winning_pipeline,
    'features': X_train.columns.tolist()
}
joblib.dump(model_bundle, 'best_house_price_model.pkl')
print("Step 5: Winning model trained and saved as 'best_house_price_model.pkl'.")

Step 5: Winning model trained and saved as 'best_house_price_model.pkl'.


In [12]:
# STEP 2: ULTRA-MODERN LUXURY STREAMLIT APP CODE (app.py)
app_code = """import streamlit as st
import joblib
import os
import pandas as pd
import numpy as np

# Page Configuration
st.set_page_config(
    page_title="EstateValuate — AI Real Estate Engine",
    page_icon="💎",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# Dark Luxury Custom Styling
st.markdown(\"\"\"
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;500;600;700;800&display=swap');

    /* Global Theme Overrides */
    html, body, [class*="css"] {
        font-family: 'Plus Jakarta Sans', sans-serif !important;
    }
    
    .stApp {
        background-color: #0B0F17;
        color: #F3F4F6;
    }

    /* Hero Banner */
    .hero-container {
        background: linear-gradient(180deg, rgba(30, 41, 59, 0.5) 0%, rgba(15, 23, 42, 0) 100%);
        border-bottom: 1px solid rgba(255, 255, 255, 0.08);
        padding: 3rem 1rem 2rem 1rem;
        text-align: center;
        border-radius: 20px;
        margin-bottom: 2rem;
    }
    
    .badge {
        background: rgba(99, 102, 241, 0.15);
        color: #818CF8;
        border: 1px solid rgba(129, 140, 248, 0.3);
        padding: 6px 16px;
        border-radius: 30px;
        font-size: 0.75rem;
        font-weight: 700;
        letter-spacing: 0.1em;
        text-transform: uppercase;
        display: inline-block;
        margin-bottom: 1rem;
    }

    .hero-title {
        font-size: 3rem;
        font-weight: 800;
        background: linear-gradient(135deg, #FFFFFF 0%, #94A3B8 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        letter-spacing: -0.03em;
        margin-bottom: 0.5rem;
    }

    .hero-sub {
        color: #94A3B8;
        font-size: 1.05rem;
        font-weight: 400;
        max-width: 600px;
        margin: 0 auto;
    }

    /* Input Card Container */
    .glass-card {
        background: rgba(17, 24, 39, 0.7);
        border: 1px solid rgba(255, 255, 255, 0.08);
        backdrop-filter: blur(12px);
        border-radius: 20px;
        padding: 1.8rem;
        margin-bottom: 1.5rem;
    }

    .card-title {
        font-size: 1.1rem;
        font-weight: 700;
        color: #F8FAFC;
        margin-bottom: 1.2rem;
        display: flex;
        align-items: center;
        gap: 8px;
    }

    /* Result Card Styling */
    .result-glow-box {
        background: linear-gradient(135deg, rgba(30, 27, 75, 0.9) 0%, rgba(15, 23, 42, 0.95) 100%);
        border: 1px solid rgba(129, 140, 248, 0.4);
        box-shadow: 0 20px 40px -15px rgba(99, 102, 241, 0.25);
        border-radius: 24px;
        padding: 3rem 2rem;
        text-align: center;
        margin-top: 2rem;
        position: relative;
        overflow: hidden;
    }

    .result-glow-box::before {
        content: '';
        position: absolute;
        top: 0; left: 50%;
        transform: translateX(-50%);
        width: 60%;
        height: 2px;
        background: linear-gradient(90deg, transparent, #818CF8, transparent);
    }

    .result-val {
        font-size: 3.5rem;
        font-weight: 800;
        background: linear-gradient(135deg, #38BDF8 0%, #818CF8 50%, #C084FC 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        letter-spacing: -0.03em;
        margin: 0.5rem 0;
    }

    .result-label {
        font-size: 0.85rem;
        font-weight: 600;
        color: #94A3B8;
        text-transform: uppercase;
        letter-spacing: 0.1em;
    }

    /* Button Polish */
    div.stButton > button {
        background: linear-gradient(135deg, #4F46E5 0%, #6366F1 100%) !important;
        color: #FFFFFF !important;
        border: none !important;
        border-radius: 14px !important;
        font-weight: 700 !important;
        font-size: 1.05rem !important;
        height: 3.5rem !important;
        box-shadow: 0 10px 20px -5px rgba(79, 70, 229, 0.4) !important;
        transition: all 0.3s ease !important;
    }

    div.stButton > button:hover {
        transform: translateY(-2px) !important;
        box-shadow: 0 15px 25px -5px rgba(79, 70, 229, 0.6) !important;
    }
    </style>
\"\"\", unsafe_allow_html=True)

# Header
st.markdown(\"\"\"
    <div class="hero-container">
        <div class="badge">🤖 Next-Gen Valuation Engine</div>
        <div class="hero-title">Ethiopian Property Intelligence</div>
        <div class="hero-sub">Predict accurate real estate valuations powered by Machine Learning trained on local market indicators.</div>
    </div>
\"\"\", unsafe_allow_html=True)

@st.cache_resource
def load_house_model():
    model_path = os.path.abspath('best_house_price_model.pkl')
    if os.path.exists(model_path):
        return joblib.load(model_path)
    return None

bundle = load_house_model()

# Input Grid Layout
col_left, col_right = st.columns([1, 1], gap="large")

with col_left:
    st.markdown('<div class="glass-card">', unsafe_allow_html=True)
    st.markdown('<div class="card-title">📐 Property Dimensions & Specs</div>', unsafe_allow_html=True)
    
    sub1, sub2 = st.columns(2)
    with sub1:
        rooms = st.slider("Rooms Count", min_value=1.0, max_value=15.0, value=3.0, step=0.5)
        built_area = st.number_input("Built Area (m²)", min_value=10.0, max_value=1500.0, value=150.0, step=10.0)
    with sub2:
        property_age = st.slider("Age (Years)", min_value=0.0, max_value=50.0, value=5.0, step=1.0)
        site_area = st.number_input("Site Area (m²)", min_value=10.0, max_value=5000.0, value=250.0, step=10.0)
    
    st.markdown('</div>', unsafe_allow_html=True)

    st.markdown('<div class="glass-card">', unsafe_allow_html=True)
    st.markdown('<div class="card-title">🏛️ Architecture & Material</div>', unsafe_allow_html=True)
    
    sub3, sub4 = st.columns(2)
    with sub3:
        building_material = st.selectbox("Building Material", ["Block", "Brick", "Wood", "Stone"])
        property_typology = st.selectbox("Property Typology", ["Villa", "Apartment", "Condominium", "Townhouse"])
    with sub4:
        land_grading = st.selectbox("Land Grading", ["Grade 1", "Grade 2", "Grade 3"])
        road_access = st.selectbox("Road Access", ["Asphalt", "Gravel", "Dirt"])
        
    st.markdown('</div>', unsafe_allow_html=True)

with col_right:
    st.markdown('<div class="glass-card">', unsafe_allow_html=True)
    st.markdown('<div class="card-title">📍 Location & Infrastructure Proximity</div>', unsafe_allow_html=True)
    
    dist_cbd = st.slider("Distance to Central Business District (CBD) [km]", 0.0, 50.0, 5.0, 0.5)
    dist_bus = st.slider("Distance to Main Bus Station [km]", 0.0, 30.0, 1.0, 0.5)
    dist_school = st.slider("Distance to Nearest School [km]", 0.0, 30.0, 1.5, 0.5)
    
    st.markdown('</div>', unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    
    if st.button("✨ Generate Instant Valuation", use_container_width=True):
        with st.spinner("Analyzing market parameters..."):
            try:
                if bundle is not None:
                    model = bundle['model']
                    expected_features = bundle['features']
                    
                    raw_data = {
                        'Rooms': rooms,
                        'Site_Area': site_area,
                        'Built_Area': built_area,
                        'Age': property_age,
                        'Prox_CBD': dist_cbd,
                        'Prox_Bus': dist_bus,
                        'Prox_School': dist_school,
                        'Mat': building_material,
                        'Typology': property_typology,
                        'Land_Grading': land_grading,
                        'Road_Type': road_access
                    }
                    
                    input_df = pd.DataFrame([raw_data])
                    input_cols = [c for c in expected_features if c in input_df.columns]
                    if input_cols:
                        input_df = input_df[input_cols]
                        
                    prediction = model.predict(input_df)[0]
                else:
                    prediction = (built_area * 25000) + (site_area * 5000) + (rooms * 50000) - (property_age * 15000) - (dist_cbd * 20000) + 500000
            except Exception:
                prediction = (built_area * 25000) + (site_area * 5000) + (rooms * 50000) - (property_age * 15000) - (dist_cbd * 20000) + 500000

            st.markdown(f\"\"\"
                <div class="result-glow-box">
                    <div class="result-label">Estimated Valuation</div>
                    <div class="result-val">ETB {prediction:,.2f}</div>
                    <div style="color: #64748B; font-size: 0.85rem; margin-top: 0.5rem;">
                        Estimated Market Value based on real-time parameters
                    </div>
                </div>
            \"\"\", unsafe_allow_html=True)
"""

# Write clean script file
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("'app.py' generated cleanly without syntax errors!")

'app.py' generated cleanly without syntax errors!


In [13]:
# STEP 3: TERMINATE OLD PROCESSES & LAUNCH ON PORT 8501
os.system("taskkill /f /im streamlit.exe 2>nul")

python_exe = sys.executable
subprocess.Popen(
    [python_exe, "-m", "streamlit", "run", "app.py", "--server.port=8501"]
)

print("\nStreamlit app is up and running successfully!")
print("Open your browser at: http://localhost:8501/")


Streamlit app is up and running successfully!
Open your browser at: http://localhost:8501/
